In [1]:
!pip install -q pymongo

In [2]:
from google.colab import userdata
from pymongo import MongoClient

In [3]:
client = MongoClient(userdata.get("MONGO_URI"))
db = client["cinema"]
print(client.admin.command("ping"))

{'ok': 1}


## Analyse 1 — Les genres
**Question :** quels genres sont les plus représentés, les mieux notés et les plus rentables ?

In [4]:
pipeline_genres = [
    {"$unwind": "$genres"},
    {"$group": {
        "_id": "$genres",
        "nb_films": {"$sum": 1},
        "note_moyenne": {"$avg": "$note_moyenne"},
        "budget_moyen": {"$avg": "$budget"},
        "recettes_moyennes": {"$avg": "$recettes"},
    }},
    {"$sort": {"nb_films": -1}},
    {"$out": "stats_genres"},
]

list(db["films"].aggregate(pipeline_genres))

for g in db["stats_genres"].find():
    print(f"{g['_id']:<20} {g['nb_films']:>5} films | note {g['note_moyenne']:.2f} "
          f"| recettes moy. {g['recettes_moyennes'] / 1e6:,.0f} M$")

Drame                 2552 films | note 6.87 | recettes moy. 63 M$
Comédie               2055 films | note 6.46 | recettes moy. 114 M$
Thriller              1701 films | note 6.43 | recettes moy. 81 M$
Action                1526 films | note 6.52 | recettes moy. 190 M$
Aventure              1068 films | note 6.63 | recettes moy. 282 M$
Romance                949 films | note 6.68 | recettes moy. 80 M$
Crime                  903 films | note 6.61 | recettes moy. 80 M$
Horreur                903 films | note 6.17 | recettes moy. 58 M$
Science-Fiction        797 films | note 6.48 | recettes moy. 221 M$
Fantastique            736 films | note 6.65 | recettes moy. 205 M$
Familial               701 films | note 6.70 | recettes moy. 232 M$
Animation              582 films | note 7.09 | recettes moy. 237 M$
Mystère                578 films | note 6.42 | recettes moy. 71 M$
Histoire               319 films | note 7.08 | recettes moy. 70 M$
Guerre                 176 films | note 7.05 | recettes

L'Aventure est la reine du box-office (282 M$), suivie de l'Animation, du Familial et de la Science-Fiction. Ce sont les genres des blockbusters.

L'Horreur a la plus mauvaise note (6,17) et peu de recettes. Mais c'est un genre réputé pour ses petits budgets, donc il peut être très rentable malgré tout.


⚠️ : des recettes élevées ne veulent pas dire rentable. Un film qui coûte 300 M$ et en rapporte 400 M$ gagne moins, qu'un film d'horreur à 5 M$ qui en rapporte 50.

## Analyse 2 — Évolution des budgets, recettes et rentabilité par année
**Question :** les films coûtent-ils et rapportent-ils de plus en plus ? Sont-ils plus ou moins rentables ?

In [5]:
pipeline_annees = [
    {"$match": {"budget": {"$ne": None}, "recettes": {"$ne": None}}},
    {"$set": {"rentabilite": {"$divide": ["$recettes", "$budget"]}}},
    {"$group": {
        "_id": "$annee",
        "nb_films": {"$sum": 1},
        "budget_moyen": {"$avg": "$budget"},
        "recettes_moyennes": {"$avg": "$recettes"},
        "rentabilite_mediane": {"$median": {"input": "$rentabilite", "method": "approximate"}},
    }},
    {"$sort": {"_id": 1}},
    {"$out": "stats_annees"},
]

list(db["films"].aggregate(pipeline_annees))

for a in db["stats_annees"].find():
    print(f"{a['_id']} | {a['nb_films']:>3} films | budget moy. {a['budget_moyen'] / 1e6:>5.1f} M$ "
          f"| recettes moy. {a['recettes_moyennes'] / 1e6:>6.1f} M$ | rentabilité médiane × {a['rentabilite_mediane']:.2f}")

2000 | 110 films | budget moy.  46.0 M$ | recettes moy.  111.9 M$ | rentabilité médiane × 2.25
2001 | 131 films | budget moy.  42.9 M$ | recettes moy.  111.5 M$ | rentabilité médiane × 1.65
2002 | 137 films | budget moy.  41.5 M$ | recettes moy.  116.1 M$ | rentabilité médiane × 2.15
2003 | 140 films | budget moy.  43.9 M$ | recettes moy.  120.2 M$ | rentabilité médiane × 2.30
2004 | 147 films | budget moy.  45.2 M$ | recettes moy.  123.9 M$ | rentabilité médiane × 2.17
2005 | 165 films | budget moy.  41.7 M$ | recettes moy.  105.3 M$ | rentabilité médiane × 2.03
2006 | 182 films | budget moy.  39.8 M$ | recettes moy.  101.5 M$ | rentabilité médiane × 2.19
2007 | 181 films | budget moy.  41.0 M$ | recettes moy.  111.1 M$ | rentabilité médiane × 2.14
2008 | 178 films | budget moy.  43.5 M$ | recettes moy.  120.2 M$ | rentabilité médiane × 2.15
2009 | 198 films | budget moy.  41.0 M$ | recettes moy.  122.3 M$ | rentabilité médiane × 1.97
2010 | 194 films | budget moy.  44.4 M$ | recettes

* Période COVID 2020 : seulement 88 films (contre environ 180 d'habitude), une
rentabilité médiane de × 0.81. Plus de la moitié des films ont perdu de l'argent, à cause des cinémas fermés.

* 2021 à 2025 : une reprise progressive de la rentabilité (1,09, puis 1,56, 1,80, 1,90, 2,31). En 2025, on retrouve presque le niveau d'avant-crise.

## Analyse 3 — Les réalisateurs les mieux notés
**Question :** quels réalisateurs ont, en moyenne, les films les mieux notés ?

In [6]:
pipeline_realisateurs = [
    {"$match": {"role": "Director"}},
    {"$lookup": {
        "from": "films",              # collection à joindre
        "localField": "film_id",      # champ dans credits
        "foreignField": "_id",        # champ correspondant dans films
        "as": "film",                 # nom du champ qui recevra le film trouvé
    }},
    {"$unwind": "$film"},
    {"$group": {
        "_id": "$personne_id",
        "nom": {"$first": "$nom"},
        "nb_films": {"$sum": 1},
        "note_moyenne": {"$avg": "$film.note_moyenne"},
        "recettes_totales": {"$sum": "$film.recettes"},
        "films": {"$push": "$film.titre"},
    }},
    {"$match": {"nb_films": {"$gte": 5}}},
    {"$sort": {"note_moyenne": -1}},
    {"$limit": 20},
    {"$out": "top_realisateurs"},
]

list(db["credits"].aggregate(pipeline_realisateurs))

for r in db["top_realisateurs"].find():
    print(f"{r['nom']:<25} {r['nb_films']:>2} films | note {r['note_moyenne']:.2f} "
          f"| recettes totales {r['recettes_totales'] / 1e9:.2f} Md$")

Hayao Miyazaki             5 films | note 7.97 | recettes totales 1.13 Md$
Christopher Nolan         11 films | note 7.90 | recettes totales 6.18 Md$
Quentin Tarantino          8 films | note 7.79 | recettes totales 2.01 Md$
Dean DeBlois               5 films | note 7.76 | recettes totales 2.57 Md$
Pete Docter                5 films | note 7.74 | recettes totales 2.36 Md$
Peter Jackson              9 films | note 7.72 | recettes totales 6.53 Md$
Hirokazu Kore-eda          6 films | note 7.70 | recettes totales 0.04 Md$
Mamoru Hosoda              6 films | note 7.68 | recettes totales 0.22 Md$
Denis Villeneuve           9 films | note 7.64 | recettes totales 1.81 Md$
Chris Sanders              5 films | note 7.64 | recettes totales 1.80 Md$
Keishi Otomo               5 films | note 7.63 | recettes totales 0.10 Md$
Hideaki Anno               5 films | note 7.61 | recettes totales 0.30 Md$
Makoto Shinkai             7 films | note 7.58 | recettes totales 0.92 Md$
Martin Scorsese          

* Hayao Miyazaki est n° 1 (7,97), juste devant Christopher Nolan (7,90).

* Nolan est le plus impressionnant : sur 11 films, il garde une note très élevée et cumule 6,18 milliards de recettes. Qualité et succès commercial à la fois.

L'animation japonaise est omniprésente : Miyazaki, Hosoda, Shinkai, Anno. C'est cohérent avec l'analyse 1, où l'Animation était l'un des genres les mieux notés.

Deux analyses différentes qui se confirment, c'est un bon signe.

Qualité et box-office sont deux choses différentes.

* Hirokazu Kore-eda (7,70) a des films très bien notés, mais seulement 0,04 Md$ de recettes.

* Peter Jackson (Le Seigneur des anneaux, Le Hobbit) cumule 6,53 Md$.


Le cinéma coréen est bien présent : Park Chan-wook et Bong Joon-ho (Parasite).